# Phase 1 - Smart Union Merging (Research-Based Approach)

## Objective
Implement the **smart union merging strategy** based on Oh et al. (2024) to detect timestamp manipulation through cross-artifact correlation of $LogFile and $UsnJrnl.

## Research Foundation
**Paper**: "Forensic Detection of Timestamp Manipulation for Digital Forensic Investigation"  
**Authors**: Oh, J., Lee, S., & Hwang, H. (2024)  
**Published**: IEEE Access, DOI: 10.1109/ACCESS.2024.3395644

## Key Detection Patterns

### Basic Detection Pattern (Page 11):
- **UsnJrnl**: BASIC_INFO_CHANGE followed by CLOSE within 0-1 second
- **LogFile**: UpdateResidentValue operation at offset 0x38 ($SI attribute)

### File System Tunneling (Page 7):
- Delete/Rename → Create within 15 seconds causes benign $SI-C changes
- Must detect and exclude to reduce false positives

### Confidence Levels:
- **HIGH**: Evidence in BOTH $LogFile AND $UsnJrnl
- **MEDIUM**: Evidence in ONE artifact only (other may be cleared/overwritten)
- **LOW**: Weak indicators requiring validation

## Smart Union Strategy

**CRITICAL**: Do NOT discard single-source events!

Three record types with source indicators:
1. `source='both'`: Matched records (HIGH confidence)
2. `source='logfile_only'`: LogFile patterns without UsnJrnl match (MEDIUM)
3. `source='usnjrnl_only'`: UsnJrnl patterns without LogFile match (MEDIUM)

## Expected Output
- **Input**: 96K LogFile + 3.1M UsnJrnl = 3.2M raw records
- **After filtering**: ~8K LogFile + ~25K UsnJrnl = ~33K filtered records
- **After smart union**: ~30K final records (99% reduction, no duplicates)

---

## 1. Setup & Configuration

In [28]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
from datetime import datetime, timedelta
import glob

warnings.filterwarnings('ignore')

print("✓ Libraries imported successfully")
print(f"Pandas version: {pd.__version__}")

✓ Libraries imported successfully
Pandas version: 2.3.2


In [29]:
# Define paths
# Use absolute path to ensure correct location regardless of where Jupyter is launched
BASE_DIR = Path('/Users/soni/Github/Digital-Detectives_Thesis')
RAW_DIR = BASE_DIR / 'data' / 'raw'
OUTPUT_DIR = BASE_DIR / 'data' / 'processed' / 'Phase 1 - Data Cleaning'

# Create output directory if needed
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("📂 Directory Configuration:")
print(f"  Base: {BASE_DIR}")
print(f"  Raw data: {RAW_DIR}")
print(f"  Output: {OUTPUT_DIR}")
print(f"\nChecking raw data directories:")
print(f"  LogFile:    {RAW_DIR / 'logfile'} {'✓' if (RAW_DIR / 'logfile').exists() else '✗'}")
print(f"  UsnJrnl:    {RAW_DIR / 'usnjrnl'} {'✓' if (RAW_DIR / 'usnjrnl').exists() else '✗'}")
print(f"  Suspicious: {RAW_DIR / 'suspicious'} {'✓' if (RAW_DIR / 'suspicious').exists() else '✗'}")

📂 Directory Configuration:
  Base: /Users/soni/Github/Digital-Detectives_Thesis
  Raw data: /Users/soni/Github/Digital-Detectives_Thesis/data/raw
  Output: /Users/soni/Github/Digital-Detectives_Thesis/data/processed/Phase 1 - Data Cleaning

Checking raw data directories:
  LogFile:    /Users/soni/Github/Digital-Detectives_Thesis/data/raw/logfile ✓
  UsnJrnl:    /Users/soni/Github/Digital-Detectives_Thesis/data/raw/usnjrnl ✓
  Suspicious: /Users/soni/Github/Digital-Detectives_Thesis/data/raw/suspicious ✓


---
## 2. Helper Functions (Research-Based)

In [ ]:
def find_basic_detection_pattern(usn_df):
    """
    Find UsnJrnl records matching the basic detection pattern:
    Records containing BASIC_INFO_CHANGE (with or without CLOSE)

    Based on: Oh et al. (2024), Section V.E.1, Page 11
    Quote: "timestamp manipulation creates a record added BASIC_INFO_CHANGE
    value in the Reason Flag"

    CRITICAL UPDATE (from diagnostic analysis):
    100% of timestomped events have 'Basic_Info_Changed / File_Closed'
    COMBINED in ONE record, not as two separate records. Therefore, we simply
    filter to any record containing 'Basic_Info_Change' rather than requiring
    a separate CLOSE event.

    Evidence from diagnostic analysis (00_Diagnostic_Analysis.ipynb):
    - 238 UsnJrnl timestomped events analyzed
    - 238 (100.0%) have BASIC_INFO_CHANGE + CLOSE combined in one record
    - 0 (0.0%) have BASIC_INFO_CHANGE only or CLOSE only as separate records

    This corrected filter should capture all 488 timestomped events across
    all 12 cases (previously missed 236 events = 48.4% data loss).

    Returns:
        DataFrame with only records containing Basic_Info_Change
    """
    print("  Finding BASIC_INFO_CHANGE pattern...")

    # Filter to records containing BASIC_INFO_CHANGE
    # This captures both:
    # 1. Combined events: "Basic_Info_Changed / File_Closed" (100% of timestomped)
    # 2. Separate events: "Basic_Info_Changed" only (edge cases if any)
    result = usn_df[
        usn_df['usn_event_info'].str.contains('Basic_Info_Change', na=False, case=False)
    ].copy()

    print(f"    ✓ BASIC_INFO_CHANGE events: {len(result):,}")

    return result

In [31]:
def filter_logfile_timestamp_changes(lf_df):
    """
    Filter LogFile to timestamp-relevant events:
    UpdateResidentValue operations targeting $SI attribute
    
    Based on: Oh et al. (2024), Section V.C, Page 8
    Quote: "a record whose Redo OP value of the record header is 
    'UpdateResidentValue' (0x7) is created when timestamp manipulation 
    is performed"
    
    Returns:
        DataFrame with only timestamp-relevant LogFile events
    """
    print("  Filtering LogFile to timestamp-relevant events...")
    
    # Keep Time Reversal events (explicit timestamp manipulation indicator)
    time_reversal = lf_df[
        lf_df['lf_event'].str.contains('Time Reversal', na=False, case=False)
    ]
    
    # Keep Update events (includes UpdateResidentValue)
    update_events = lf_df[
        lf_df['lf_event'].str.contains('Update', na=False, case=False)
    ]
    
    # Combine and remove duplicates
    result = pd.concat([time_reversal, update_events]).drop_duplicates()
    
    print(f"    Time Reversal: {len(time_reversal):,} events")
    print(f"    Update: {len(update_events):,} events")
    print(f"    ✓ Total filtered: {len(result):,} events")
    
    return result

In [32]:
def detect_file_system_tunneling(merged_df, all_usn_events):
    """
    Detect file system tunneling to reduce false positives.
    
    File system tunneling: Windows caches filename and $SI-C when file is 
    deleted/renamed/moved, then applies cached values to new file with same 
    name within 15 seconds.
    
    Based on: Oh et al. (2024), Section V.D.3, Algorithm 4, Page 7
    Quote: "within a specific time (default: 15 seconds)"
    
    Args:
        merged_df: DataFrame with merged records
        all_usn_events: Complete UsnJrnl dataset for context lookup
    
    Returns:
        DataFrame with 'is_tunneling' column added
    """
    print("  Detecting file system tunneling patterns...")
    
    merged_df['is_tunneling'] = False
    tunneling_count = 0
    
    for idx, row in merged_df.iterrows():
        # Look for delete/rename/move events within 15 seconds BEFORE this event
        time_window_start = row['eventtime_dt'] - timedelta(seconds=15)
        time_window_end = row['eventtime_dt']
        
        # Check for suspicious prior events on same file
        prior_events = all_usn_events[
            (all_usn_events['merge_key'] == row['merge_key']) &
            (all_usn_events['eventtime_dt'] >= time_window_start) &
            (all_usn_events['eventtime_dt'] < time_window_end)
        ]
        
        # Check for delete/rename/move patterns
        tunneling_indicators = prior_events[
            prior_events['usn_event_info'].str.contains(
                'File_Delete|Rename_Old_Name|Rename_New_Name', 
                na=False, 
                case=False, 
                regex=True
            )
        ]
        
        if len(tunneling_indicators) > 0:
            merged_df.at[idx, 'is_tunneling'] = True
            tunneling_count += 1
    
    print(f"    ✓ Tunneling detected: {tunneling_count:,} events ({tunneling_count/len(merged_df)*100:.2f}%)")
    return merged_df

In [33]:
def detect_timestomping_tool(merged_df, lf_raw, usn_raw, time_window=10):
    """
    Detect which timestomping tool was likely used based on program execution 
    evidence within a time window before the manipulation.
    
    This function searches for prefetch files (.pf) or executable launches of 
    known timestomping tools within ±10 seconds of the manipulation event.
    
    Known tools:
    - NewFileTime / NEWFILETIME
    - SetMACE / SETMACE
    - NTimestomp / NTIMESTOMP
    - Timestomp / TIMESTOMP
    - PowerShell (Set-ItemProperty, [IO.File]::SetCreationTime)
    
    Args:
        merged_df: DataFrame with merged timestamp manipulation records
        lf_raw: Complete raw LogFile data (for tool execution lookups)
        usn_raw: Complete raw UsnJrnl data (for tool execution lookups)
        time_window: Seconds before/after manipulation to search (default: 10)
    
    Returns:
        DataFrame with 'suspected_tool' column added
    """
    print("  Detecting timestomping tools...")
    
    # Known tool patterns (case-insensitive)
    tool_patterns = {
        'NewFileTime': r'newfiletime|new.*file.*time',
        'SetMACE': r'setmace|set.*mace',
        'NTimestomp': r'ntimestomp|ntimes',
        'Timestomp': r'timestomp(?!er)',  # Match 'timestomp' but not 'timestomper'
        'PowerShell': r'powershell\.exe|pwsh\.exe'
    }
    
    merged_df['suspected_tool'] = 'Unknown'
    tool_detections = {}
    
    for idx, row in merged_df.iterrows():
        if row['is_timestomped'] == 0:
            continue  # Only check timestomped records
        
        # Define search window
        time_start = row['eventtime_dt'] - timedelta(seconds=time_window)
        time_end = row['eventtime_dt'] + timedelta(seconds=time_window)
        
        # Search in LogFile for File Creation or program execution events
        lf_nearby = lf_raw[
            (lf_raw['eventtime_dt'] >= time_start) &
            (lf_raw['eventtime_dt'] <= time_end)
        ]
        
        # Search in UsnJrnl for File_Created events
        usn_nearby = usn_raw[
            (usn_raw['eventtime_dt'] >= time_start) &
            (usn_raw['eventtime_dt'] <= time_end)
        ]
        
        # Check for tool signatures
        detected_tool = 'Unknown'
        
        for tool_name, pattern in tool_patterns.items():
            # Check LogFile filenames
            lf_matches = lf_nearby[
                lf_nearby['filename'].str.contains(pattern, case=False, regex=True, na=False)
            ]
            
            # Check UsnJrnl filenames
            usn_matches = usn_nearby[
                usn_nearby['filename'].str.contains(pattern, case=False, regex=True, na=False)
            ]
            
            if len(lf_matches) > 0 or len(usn_matches) > 0:
                detected_tool = tool_name
                tool_detections[tool_name] = tool_detections.get(tool_name, 0) + 1
                break  # Take first match
        
        merged_df.at[idx, 'suspected_tool'] = detected_tool
    
    # Print summary
    total_timestomped = (merged_df['is_timestomped'] == 1).sum()
    tools_detected = sum(tool_detections.values())
    
    print(f"    Timestomped events: {total_timestomped}")
    print(f"    Tools detected: {tools_detected}")
    
    if tool_detections:
        for tool, count in sorted(tool_detections.items(), key=lambda x: x[1], reverse=True):
            print(f"      - {tool}: {count}")
    
    unknown_count = (merged_df['suspected_tool'] == 'Unknown').sum() - (len(merged_df) - total_timestomped)
    if unknown_count > 0:
        print(f"      - Unknown: {unknown_count}")
    
    return merged_df

---
## 3. Discover Available Cases

In [34]:
print("=" * 80)
print("DISCOVERING CASES")
print("=" * 80)

# Find all LogFile CSV files (one per case)
logfile_files = sorted(glob.glob(str(RAW_DIR / 'logfile' / '*.csv')))
print(f"\nFound {len(logfile_files)} LogFile cases:")
for f in logfile_files[:3]:  # Show first 3
    print(f"  - {Path(f).name}")
if len(logfile_files) > 3:
    print(f"  ... and {len(logfile_files) - 3} more")

# Extract case IDs from filenames
case_ids = []
for f in logfile_files:
    filename = Path(f).stem
    num_str = ''.join([c for c in filename if c.isdigit()][:2])  # Take first 2 digits
    if num_str:
        case_ids.append(int(num_str))

case_ids = sorted(list(set(case_ids)))
print(f"\n✓ Detected {len(case_ids)} cases: {case_ids}")

DISCOVERING CASES

Found 12 LogFile cases:
  - 01-PE-LogFile.csv
  - 02-PE-LogFile.csv
  - 03-PE-LogFile.csv
  ... and 9 more

✓ Detected 12 cases: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]


---
## 4. Process Each Case with Smart Union Strategy

### Processing Steps (per case):

1. **Load** LogFile, UsnJrnl, and Suspicious labels
2. **Filter** to detection patterns:
   - LogFile: UpdateResidentValue, Time Reversal
   - UsnJrnl: BASIC_INFO_CHANGE + CLOSE pattern
3. **Match** with 1-second window (cross-artifact validation)
4. **Separate** unmatched records:
   - `logfile_only`: LogFile patterns without UsnJrnl match
   - `usnjrnl_only`: UsnJrnl patterns without LogFile match
5. **Union** all three types with source indicators
6. **Detect** file system tunneling (15-second window)
7. **Apply** labels from suspicious.csv
8. **Detect** timestomping tools (lightweight lookup in raw data)
9. **Save** case file

In [35]:
print("=" * 80)
print("PROCESSING CASES WITH SMART UNION STRATEGY")
print("=" * 80)

case_stats = []

# Process all cases
TEST_MODE = False
cases_to_process = [case_ids[0]] if TEST_MODE else case_ids

if TEST_MODE:
    print("\n⚠️  TEST MODE: Processing only Case 1")
    print("    Change TEST_MODE = False to process all cases\n")

for case_id in cases_to_process:
    print(f"\n{'=' * 80}")
    print(f"CASE {case_id}")
    print("=" * 80)
    
    # ==========================================
    # STEP 1: Load Files
    # ==========================================
    case_pattern = f"{case_id:02d}"
    lf_file = list((RAW_DIR / 'logfile').glob(f'{case_pattern}*.csv'))[0]
    usn_file = list((RAW_DIR / 'usnjrnl').glob(f'{case_pattern}*.csv'))[0]
    sus_file = list((RAW_DIR / 'suspicious').glob(f'{case_pattern}*.csv'))[0]
    
    print(f"\n📂 Loading files:")
    print(f"  LogFile: {lf_file.name}")
    print(f"  UsnJrnl: {usn_file.name}")
    print(f"  Suspicious: {sus_file.name}")
    
    print(f"\n[1/9] Loading data...")
    lf_df = pd.read_csv(lf_file, encoding='utf-8-sig', low_memory=False)
    usn_df = pd.read_csv(usn_file, encoding='utf-8-sig', low_memory=False)
    sus_df = pd.read_csv(sus_file, encoding='utf-8-sig')
    
    print(f"  LogFile: {len(lf_df):,} records")
    print(f"  UsnJrnl: {len(usn_df):,} records")
    print(f"  Suspicious: {len(sus_df):,} labels")
    
    # Rename columns to standardized names
    lf_df = lf_df.rename(columns={
        'LSN': 'lf_lsn',
        'EventTime(UTC+8)': 'eventtime',
        'Event': 'lf_event',
        'Detail': 'lf_detail',
        'File/Directory Name': 'filename',
        'Full Path': 'filepath',
        'CreationTime': 'lf_creation_time',
        'ModifiedTime': 'lf_modified_time',
        'MFTModifiedTime': 'lf_mft_modified_time',
        'AccessedTime': 'lf_accessed_time',
        'Redo': 'lf_redo',
        'Target VCN': 'lf_target_vcn',
        'Cluster Index': 'lf_cluster_index'
    })
    
    usn_df = usn_df.rename(columns={
        'TimeStamp(UTC+8)': 'eventtime',
        'USN': 'usn_usn',
        'File/Directory Name': 'filename',
        'FullPath': 'filepath',
        'EventInfo': 'usn_event_info',
        'SourceInfo': 'usn_source_info',
        'FileAttribute': 'usn_file_attribute',
        'Carving Flag': 'usn_carving_flag',
        'FileReferenceNumber': 'usn_file_reference_number',
        'ParentFileReferenceNumber': 'usn_parent_file_reference_number'
    })
    
    # Add case_id
    lf_df['case_id'] = case_id
    usn_df['case_id'] = case_id
    
    # Parse timestamps
    lf_df['eventtime_dt'] = pd.to_datetime(lf_df['eventtime'], errors='coerce')
    usn_df['eventtime_dt'] = pd.to_datetime(usn_df['eventtime'], errors='coerce')
    
    # Create merge keys
    lf_df['merge_key'] = (lf_df['filepath'].fillna('').astype(str) + '|' + 
                          lf_df['filename'].fillna('').astype(str))
    usn_df['merge_key'] = (usn_df['filepath'].fillna('').astype(str) + '|' + 
                           usn_df['filename'].fillna('').astype(str))
    
    # Keep original DataFrames for tool detection and tunneling
    lf_df_full = lf_df.copy()
    usn_df_full = usn_df.copy()
    
    # ==========================================
    # STEP 2: Filter to Detection Patterns
    # ==========================================
    print(f"\n[2/9] Filtering to detection patterns...")
    
    lf_filtered = filter_logfile_timestamp_changes(lf_df)
    usn_filtered = find_basic_detection_pattern(usn_df)
    
    print(f"  Summary: {len(lf_df):,} → {len(lf_filtered):,} LogFile events")
    print(f"  Summary: {len(usn_df):,} → {len(usn_filtered):,} UsnJrnl events")
    
    # ==========================================
    # STEP 3: Match with 1-Second Window
    # ==========================================
    print(f"\n[3/9] Matching LogFile ↔ UsnJrnl (±1 second window)...")
    
    matched_records = []
    matched_lf_indices = set()
    matched_usn_indices = set()
    
    for lf_idx, lf_row in lf_filtered.iterrows():
        # Find UsnJrnl events for same file within ±1 second
        potential_matches = usn_filtered[
            (usn_filtered['merge_key'] == lf_row['merge_key']) &
            (usn_filtered['eventtime_dt'] >= lf_row['eventtime_dt'] - timedelta(seconds=1)) &
            (usn_filtered['eventtime_dt'] <= lf_row['eventtime_dt'] + timedelta(seconds=1))
        ]
        
        if len(potential_matches) > 0:
            # Take closest match
            potential_matches['time_diff'] = (potential_matches['eventtime_dt'] - lf_row['eventtime_dt']).abs()
            closest = potential_matches.nsmallest(1, 'time_diff').iloc[0]
            
            # Merge records
            merged_row = lf_row.copy()
            for col in usn_filtered.columns:
                if col not in merged_row.index and col not in ['eventtime', 'eventtime_dt', 'merge_key', 'case_id', 'filename', 'filepath']:
                    merged_row[col] = closest[col]
            
            merged_row['source'] = 'both'
            merged_row['time_diff_seconds'] = closest['time_diff'].total_seconds()
            matched_records.append(merged_row)
            
            matched_lf_indices.add(lf_idx)
            matched_usn_indices.add(closest.name)
    
    matched_df = pd.DataFrame(matched_records) if matched_records else pd.DataFrame()
    print(f"  ✓ Matched: {len(matched_df):,} records (source='both')")
    
    # ==========================================
    # STEP 4: Separate Unmatched LogFile Records
    # ==========================================
    print(f"\n[4/9] Extracting unmatched LogFile records...")
    
    lf_only = lf_filtered[~lf_filtered.index.isin(matched_lf_indices)].copy()
    lf_only['source'] = 'logfile_only'
    lf_only['time_diff_seconds'] = np.nan
    
    print(f"  ✓ LogFile-only: {len(lf_only):,} records (source='logfile_only')")
    
    # ==========================================
    # STEP 5: Separate Unmatched UsnJrnl Records
    # ==========================================
    print(f"\n[5/9] Extracting unmatched UsnJrnl records...")
    
    usn_only = usn_filtered[~usn_filtered.index.isin(matched_usn_indices)].copy()
    usn_only['source'] = 'usnjrnl_only'
    usn_only['time_diff_seconds'] = np.nan
    
    print(f"  ✓ UsnJrnl-only: {len(usn_only):,} records (source='usnjrnl_only')")
    
    # ==========================================
    # STEP 6: Smart Union (Combine All Three)
    # ==========================================
    print(f"\n[6/9] Creating smart union...")
    
    # Combine all three types
    final_df = pd.concat([matched_df, lf_only, usn_only], ignore_index=True)
    
    print(f"  ✓ Total records: {len(final_df):,}")
    print(f"    - both: {(final_df['source'] == 'both').sum():,}")
    print(f"    - logfile_only: {(final_df['source'] == 'logfile_only').sum():,}")
    print(f"    - usnjrnl_only: {(final_df['source'] == 'usnjrnl_only').sum():,}")
    
    # ==========================================
    # STEP 7: Detect File System Tunneling
    # ==========================================
    print(f"\n[7/9] Detecting file system tunneling...")
    
    final_df = detect_file_system_tunneling(final_df, usn_df_full)
    
    # ==========================================
    # STEP 8: Apply Labels
    # ==========================================
    print(f"\n[8/9] Applying labels...")
    
    final_df['is_timestomped'] = 0
    
    timestomp_labels = sus_df[sus_df['category'] == 'Timestamp Manipulation']
    
    # Match by USN for UsnJrnl-sourced records
    for _, label in timestomp_labels.iterrows():
        if label['source'] == 'usnjrnl':
            mask = final_df['usn_usn'] == label['lsn/usn']
            final_df.loc[mask, 'is_timestomped'] = 1
    
    # Match by LSN for LogFile-sourced records
    for _, label in timestomp_labels.iterrows():
        if label['source'] == 'logfile':
            mask = final_df['lf_lsn'] == label['lsn/usn']
            final_df.loc[mask, 'is_timestomped'] = 1
    
    timestomped_count = final_df['is_timestomped'].sum()
    
    print(f"  Timestomp labels: {len(timestomp_labels):,}")
    print(f"  ✓ Matched timestomped: {timestomped_count}")
    
    # ==========================================
    # STEP 9: Detect Timestomping Tools
    # ==========================================
    print(f"\n[9/9] Detecting timestomping tools...")
    
    final_df = detect_timestomping_tool(final_df, lf_df_full, usn_df_full, time_window=10)
    
    # Save case file
    print(f"\n💾 Saving case file...")
    output_file = OUTPUT_DIR / f'case_{case_id}_merged.csv'
    final_df.to_csv(output_file, index=False, encoding='utf-8-sig')
    file_size = output_file.stat().st_size / (1024 * 1024)
    
    print(f"  ✓ Saved: {output_file.name}")
    print(f"  Size: {file_size:.2f} MB")
    
    # Store stats
    tools_detected = (final_df['suspected_tool'] != 'Unknown').sum()
    case_stats.append({
        'case_id': case_id,
        'logfile_raw': len(lf_df_full),
        'usnjrnl_raw': len(usn_df_full),
        'logfile_filtered': len(lf_filtered),
        'usnjrnl_filtered': len(usn_filtered),
        'matched_both': len(matched_df),
        'logfile_only': len(lf_only),
        'usnjrnl_only': len(usn_only),
        'total_merged': len(final_df),
        'tunneling_detected': final_df['is_tunneling'].sum(),
        'timestomped': timestomped_count,
        'tools_detected': tools_detected,
        'file_size_mb': file_size
    })

print(f"\n\n{'=' * 80}")
print("✅ PROCESSING COMPLETE")
print("=" * 80)

PROCESSING CASES WITH SMART UNION STRATEGY

CASE 1

📂 Loading files:
  LogFile: 01-PE-LogFile.csv
  UsnJrnl: 01-PE-UsnJrnl.csv
  Suspicious: 01-PE-Suspicious.csv

[1/9] Loading data...
  LogFile: 39,077 records
  UsnJrnl: 316,817 records
  Suspicious: 4 labels

[2/9] Filtering to detection patterns...
  Filtering LogFile to timestamp-relevant events...
    Time Reversal: 1,235 events
    Update: 0 events
    ✓ Total filtered: 1,235 events
  Finding BASIC_INFO_CHANGE + CLOSE pattern...
    BASIC_INFO_CHANGE: 24,002 events
    CLOSE: 146,395 events
    ✓ Pattern matches: 21,871 events
  Summary: 39,077 → 1,235 LogFile events
  Summary: 316,817 → 21,871 UsnJrnl events

[3/9] Matching LogFile ↔ UsnJrnl (±1 second window)...
  ✓ Matched: 1,053 records (source='both')

[4/9] Extracting unmatched LogFile records...
  ✓ LogFile-only: 182 records (source='logfile_only')

[5/9] Extracting unmatched UsnJrnl records...
  ✓ UsnJrnl-only: 20,838 records (source='usnjrnl_only')

[6/9] Creating smart 

---
## 5. Summary Statistics

In [36]:
# Create summary DataFrame
summary_df = pd.DataFrame(case_stats)

print("\n📊 SMART UNION PROCESSING SUMMARY")
print("=" * 80)
print(summary_df.to_string(index=False))

print(f"\n\n📈 TOTALS:")
print(f"  Raw records:")
print(f"    LogFile: {summary_df['logfile_raw'].sum():,}")
print(f"    UsnJrnl: {summary_df['usnjrnl_raw'].sum():,}")
print(f"    Combined: {summary_df['logfile_raw'].sum() + summary_df['usnjrnl_raw'].sum():,}")
print(f"\n  After filtering to detection patterns:")
print(f"    LogFile: {summary_df['logfile_filtered'].sum():,}")
print(f"    UsnJrnl: {summary_df['usnjrnl_filtered'].sum():,}")
print(f"    Combined: {summary_df['logfile_filtered'].sum() + summary_df['usnjrnl_filtered'].sum():,}")
print(f"\n  Smart union output:")
print(f"    Matched (both): {summary_df['matched_both'].sum():,}")
print(f"    LogFile-only: {summary_df['logfile_only'].sum():,}")
print(f"    UsnJrnl-only: {summary_df['usnjrnl_only'].sum():,}")
print(f"    Total merged: {summary_df['total_merged'].sum():,}")
print(f"\n  Detection results:")
print(f"    Tunneling detected: {summary_df['tunneling_detected'].sum():,}")
print(f"    Timestomped: {summary_df['timestomped'].sum():,}")
print(f"    Tools detected: {summary_df['tools_detected'].sum():,}")
print(f"\n  Data reduction:")
raw_total = summary_df['logfile_raw'].sum() + summary_df['usnjrnl_raw'].sum()
final_total = summary_df['total_merged'].sum()
reduction = (1 - final_total / raw_total) * 100
print(f"    {raw_total:,} → {final_total:,} records ({reduction:.1f}% reduction)")
print(f"\n  Total file size: {summary_df['file_size_mb'].sum():.2f} MB")

# Save summary
summary_file = OUTPUT_DIR / 'smart_union_summary.csv'
summary_df.to_csv(summary_file, index=False)
print(f"\n✓ Summary saved: {summary_file.name}")


📊 SMART UNION PROCESSING SUMMARY
 case_id  logfile_raw  usnjrnl_raw  logfile_filtered  usnjrnl_filtered  matched_both  logfile_only  usnjrnl_only  total_merged  tunneling_detected  timestomped  tools_detected  file_size_mb
       1        39077       316817              1235             21871          1053           182         20838         22073                 242            2               1      9.192096
       2        14783       247386                97             15045            90             7         14957         15054                 164            1               0      7.729347
       3        24063       245425                97             14977            90             7         14889         14986                 164            2               1      7.708421
       4        12731       263451                79              4370            76             3          4294          4373                  52            2               0      1.700336
       5        

---
## 6. Combine All Cases into Master Dataset

In [37]:
print("\n" + "=" * 80)
print("COMBINING ALL CASES INTO MASTER DATASET")
print("=" * 80)

# Find all case files
case_files = sorted(glob.glob(str(OUTPUT_DIR / 'case_*_merged.csv')))
print(f"\nFound {len(case_files)} case files to combine")

# Load and combine all cases
all_cases = []
for case_file in case_files:
    case_num = Path(case_file).stem.split('_')[1]
    print(f"  Loading Case {case_num}...", end=' ')
    
    df = pd.read_csv(case_file)
    all_cases.append(df)
    print(f"{len(df):,} records")

# Combine all cases
master_df = pd.concat(all_cases, ignore_index=True)

print(f"\n✓ Combined {len(case_files)} cases")
print(f"  Total records: {len(master_df):,}")
print(f"  Total timestomped: {(master_df['is_timestomped'] == 1).sum()}")
print(f"  Cases represented: {sorted(master_df['case_id'].unique().tolist())}")

# Save master dataset
master_file = OUTPUT_DIR / 'all_cases_combined.csv'
master_df.to_csv(master_file, index=False, encoding='utf-8-sig')
master_size = master_file.stat().st_size / (1024 * 1024)

print(f"\n💾 Master dataset saved:")
print(f"  File: {master_file.name}")
print(f"  Size: {master_size:.2f} MB")
print(f"  Records: {len(master_df):,}")

print("\n" + "=" * 80)
print("✅ MASTER DATASET CREATED")
print("=" * 80)


COMBINING ALL CASES INTO MASTER DATASET

Found 12 case files to combine
  Loading Case 10... 15,752 records
  Loading Case 11... 4,699 records
  Loading Case 12... 4,758 records
  Loading Case 1... 22,073 records
  Loading Case 2... 15,054 records
  Loading Case 3... 14,986 records
  Loading Case 4... 4,373 records
  Loading Case 5... 4,713 records
  Loading Case 6... 4,720 records
  Loading Case 7... 15,538 records
  Loading Case 8... 15,533 records
  Loading Case 9... 15,733 records

✓ Combined 12 cases
  Total records: 137,932
  Total timestomped: 252
  Cases represented: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]

💾 Master dataset saved:
  File: all_cases_combined.csv
  Size: 65.41 MB
  Records: 137,932

✅ MASTER DATASET CREATED


## 🚀 Next: Phase 2 - Feature Engineering

### Input:
- **Master dataset**: `all_cases_combined.csv` (~300K records across 12 cases)
- **Per-case datasets**: `case_1_merged.csv` through `case_12_merged.csv`

### Key Tasks:

#### 1. **Column Cleanup**
- Remove empty columns: `usn_carving_flag`, `lf_creation_time`, `lf_modified_time`, `lf_accessed_time`, `lf_mft_modified_time`
- These timestamp columns are empty for Time Reversal events (expected behavior)
- Actual timestamp manipulation details are in `lf_detail` column

#### 2. **Parse `lf_detail` for Timestamp Manipulation**
Extract before/after timestamps from Time Reversal events:
ModifiedTime : 2023-12-23 00:14:23 -> 2000-01-01 08:00:00(Zero in 100-nanoseconds)
Create features:
- `timestamp_before`: Original timestamp
- `timestamp_after`: Manipulated timestamp
- `timestamp_changed_to_past`: Boolean (after < before)
- `timestamp_changed_to_future`: Boolean (after > before)
- `zero_in_nanoseconds`: Boolean (indicator of tool-based manipulation)

#### 3. **Temporal Anomaly Features**
- Time difference between event occurrence and recorded timestamp
- Impossible timestamp sequences (e.g., modified time before creation time)
- Timestamp ordering violations within same file
- Distance to nearest valid timestamp

#### 4. **Cross-Artifact Consistency Features**
- Agreement/disagreement between LogFile and UsnJrnl timestamps
- Time difference between cross-artifact events (`time_diff_seconds`)
- Source-based confidence encoding (`both`=2, `logfile_only`=1, `usnjrnl_only`=1)

#### 5. **Behavioral Pattern Features**
- Event frequency per file (multiple timestamp changes)
- Temporal clustering (multiple files changed within short time window)
- Tool signature features (based on `suspected_tool`)
- File system tunneling indicator (`is_tunneling`)

#### 6. **Event-Level Features**
- Event type encoding (Time Reversal, Update)
- File path depth
- File extension
- Filepath patterns (system folders, user folders, temp folders)

### Expected Output:
- **Feature matrix**: ~80-100 features per record
- **Feature documentation**: Description of each feature and its forensic significance
- **ML-ready dataset**: Cleaned, engineered, ready for model training

### Target Features for ML Model:
- All engineered features above
- Label: `is_timestomped` (0=benign, 1=timestomped)
- Auxiliary: `case_id`, `suspected_tool`, `source` (for stratification and analysis)